# 6.1 性能优化

为Agent添加缓存层+异步处理+模型路由, benchmark对比优化前后

In [ ]:
import sys, os
sys.path.insert(0, "..")
from config import get_client; client = get_client()
from agent_project import *
from agent_project.tools import create_default_registry
print(f"LLM: {client.name} | project modules loaded")


In [ ]:
import time, hashlib
from agent_project.hybrid_search import HybridSearcher
from agent_project.pipeline import DocumentPipeline

# Baseline: build search index
docs = [f"Document {i}: This is knowledge base entry number {i} about AI and LLM technologies." for i in range(50)]
searcher = HybridSearcher()
searcher.index(docs)
print(f"Indexed {len(docs)} docs")


In [ ]:
print("===== Cache Implementation =====\n")
cache = {}
hits = misses = 0
queries = ["Document 5", "Document 5", "AI technology", "Document 5", "LLM knowledge"]
for q in queries:
    key = hashlib.md5(q.encode()).hexdigest()
    if key in cache:
        hits += 1; print(f"  CACHE HIT: {q}")
    else:
        misses += 1
        results = searcher.search(q, top_k=2)
        cache[key] = results
        print(f"  CACHE MISS: {q} -> {len(results)} results")
print(f"\nHit rate: {hits}/{hits+misses} = {hits/(hits+misses):.0%}")


In [ ]:
print("===== Async Benchmark =====\n")
import asyncio
async def search_async(q):
    await asyncio.sleep(0.01); return searcher.search(q, top_k=3)
async def batch_search(queries):
    tasks = [search_async(q) for q in queries]
    return await asyncio.gather(*tasks)
t0 = time.time()
results = asyncio.run(batch_search(queries[:5]))
print(f"Async batch (5 queries): {(time.time()-t0)*1000:.0f}ms")
t0 = time.time()
for q in queries[:5]: searcher.search(q, top_k=3)
print(f"Sync sequential (5 queries): {(time.time()-t0)*1000:.0f}ms")
